# O ano da Copa em cada lance

## Task 2: trazer o ano da Copa para cada lance

## O problema

O feed não tem o ano da Copa; as partidas têm. Join pelo `match_id`.

In [1]:
import sys
sys.path.insert(0, "../../src")

from pyspark.sql import SparkSession, functions as F

spark = (
    SparkSession.builder
    .appName("arena-broadcast")
    .master("local[*]")
    .config("spark.sql.warehouse.dir", "../../data/warehouse")
    .getOrCreate()
)

RAW = "../../data/raw"
EVENTOS = "../../data/events"

spark

A célula abaixo já vem pronta: o código do vídeo 4.1 (lances e partidas).

In [2]:
# vídeo 4.1: os lances e as partidas
from wc_pipeline.schemas import MATCHES
from wc_pipeline.transformacoes import limpar_partidas

eventos = spark.read.parquet(EVENTOS).withColumnRenamed("MatchID", "match_id")

partidas = limpar_partidas(
    spark.read.option("header", "true").schema(MATCHES).csv(f"{RAW}/WorldCupMatches.csv")
)

## Trazendo o ano

In [3]:
eventos.show(5)

+--------+--------+------+------+----------+------+---------+
|event_id|match_id|period|minute|event_type|  team|player_id|
+--------+--------+------+------+----------+------+---------+
|       1|       1|    1H|    19|      Goal|France|     5470|
|       2|       1|    1H|    40|      Goal|France|    99087|
|       3|       1|    1H|    43|      Goal|France|    60620|
|       4|       1|    2H|    70|      Goal|Mexico|    94135|
|       5|       1|    2H|    87|      Goal|France|    60620|
+--------+--------+------+------+----------+------+---------+
only showing top 5 rows


In [5]:
lances = eventos.join(partidas.select("match_id", "ano"), on="match_id")
lances.show(5)

+--------+--------+------+------+----------+------+---------+----+
|match_id|event_id|period|minute|event_type|  team|player_id| ano|
+--------+--------+------+------+----------+------+---------+----+
|       1|       1|    1H|    19|      Goal|France|     5470|1930|
|       1|       2|    1H|    40|      Goal|France|    99087|1930|
|       1|       3|    1H|    43|      Goal|France|    60620|1930|
|       1|       4|    2H|    70|      Goal|Mexico|    94135|1930|
|       1|       5|    2H|    87|      Goal|France|    60620|1930|
+--------+--------+------+------+----------+------+---------+----+
only showing top 5 rows


## Juntar tabelas pode custar caro

Para juntar pela chave, as linhas de mesmo `match_id` dos dois lados precisam estar na mesma partição. O jeito geral é
redistribuir os dois lados pela chave: o **shuffle**, que aparece no plano como **`Exchange`** (vídeo 4.3).

Quando um lado é **pequeno**, o Spark pode mandar uma **cópia inteira** dele para cada executor, e o lado grande não
sai do lugar: o **broadcast join**.

In [7]:
lances.explain(mode="formatted")

== Physical Plan ==
AdaptiveSparkPlan (12)
+- Project (11)
   +- BroadcastHashJoin Inner BuildRight (10)
      :- Project (3)
      :  +- Filter (2)
      :     +- Scan parquet  (1)
      +- BroadcastExchange (9)
         +- HashAggregate (8)
            +- Exchange (7)
               +- HashAggregate (6)
                  +- Filter (5)
                     +- Scan csv  (4)


(1) Scan parquet 
Output [7]: [event_id#0L, MatchID#1, period#2, minute#3, event_type#4, team#5, player_id#6L]
Batched: true
Location: InMemoryFileIndex [file:/home/jovyan/work/data/events]
PushedFilters: [IsNotNull(MatchID)]
ReadSchema: struct<event_id:bigint,MatchID:int,period:string,minute:int,event_type:string,team:string,player_id:bigint>

(2) Filter
Input [7]: [event_id#0L, MatchID#1, period#2, minute#3, event_type#4, team#5, player_id#6L]
Condition : isnotnull(MatchID#1)

(3) Project
Output [7]: [event_id#0L, MatchID#1 AS match_id#8, period#2, minute#3, event_type#4, team#5, player_id#6L]
Input [7]: [event_

Procure no plano:

- o tipo de join;
- se o caminho dos lances (`FileScan parquet`) tem algum `Exchange` até o join;
- o **`BroadcastExchange`** do lado das partidas.

## Por que o Spark escolheu o broadcast?

O Spark faz broadcast sozinho quando estima que um lado tem até `spark.sql.autoBroadcastJoinThreshold` (padrão: 10 MB).

In [8]:
print("limite do broadcast automático:", spark.conf.get("spark.sql.autoBroadcastJoinThreshold"))

limite do broadcast automático: 10485760b


Com os dois lados grandes, o Spark usa o **sort merge join**, com `Exchange` dos dois lados.

## Nenhum lance se perdeu?

O join padrão é o `inner`: um lance sem partida correspondente sumiria sem aviso.

In [9]:
print("lances no feed:", eventos.count())
print("lances com o ano:", lances.count())
lances.select("ano", "match_id", "minute", "event_type", "team").show(5)

lances no feed: 195747
lances com o ano: 195747
+----+--------+------+----------+------+
| ano|match_id|minute|event_type|  team|
+----+--------+------+----------+------+
|1930|       1|    19|      Goal|France|
|1930|       1|    40|      Goal|France|
|1930|       1|    43|      Goal|France|
|1930|       1|    70|      Goal|Mexico|
|1930|       1|    87|      Goal|France|
+----+--------+------+----------+------+
only showing top 5 rows


## Para consultar depois

- **Pedindo o broadcast:** `eventos.join(F.broadcast(partidas), ...)` marca um lado para broadcast, mesmo acima do
  limite. Use só quando tiver certeza de que o lado cabe na memória de cada executor.
- **Desligando o automático:** `spark.sql.autoBroadcastJoinThreshold = -1`.
- **Em SQL:** o mesmo pedido é a dica `SELECT /*+ BROADCAST(p) */ ...`.

## Encerrando

Ao terminar o notebook, encerre a sessão Spark. Isso libera a memória e a porta da Spark UI para o próximo notebook.

In [10]:
spark.stop()